# Benchmark multi-cultivo controlado por fuga de datos + Índice de Sensibilidad al Leakage (LSI)

Notebook para **Kaggle con GPU** (o Colab GPU). Reproduce todo el pipeline del estudio:
datos multi-cultivo → embeddings **DINOv2** + **pHash** → clustering a varios umbrales τ →
splits aleatorio vs **VC-GKF** → grilla de entrenamiento (4 modelos × 2 splits × 10 semillas) →
**LSI** (puntual y de curva) → evaluación **OOD** cross-dataset → **muestreo Redundancy-Aware** →
XAI ratio de energía → calibración (ECE). Guarda todos los CSV en `RESULTS/`.

**Orden de ejecución:** celdas 0–8 (setup, datos, embeddings, clustering, modelos). Luego los
experimentos A–F. Descarga los CSV de `RESULTS/` para redactar el artículo con los números reales.

> Marcadores `# TODO`: solo rutas/keys de datasets. En Kaggle activa *Internet: ON* y *GPU*.


## 0. Setup e instalación

In [ ]:
import torch, os
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# En Kaggle casi todo viene preinstalado; aseguramos lo específico:
!pip -q install timm imagehash kagglehub scikit-learn scipy pandas 2>/dev/null
print("deps OK")

## 1. Configuración global

In [ ]:
import numpy as np, random
SEEDS       = [42, 0, 1, 7, 123, 77, 88, 99, 101, 2024]   # 10 semillas -> significancia real
MODELS      = ["resnet50", "mobilenetv3", "efficientnetv2", "vit"]
TAUS        = [0.05, 0.10, 0.15, 0.20, 0.30]              # barrido para la curva LSI
IMG, BATCH, EPOCHS, LR, WD, GAMMA = 224, 32, 10, 1e-4, 1e-4, 2.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
RESULTS="/kaggle/working/RESULTS" if os.path.isdir("/kaggle/working") else "RESULTS"
os.makedirs(RESULTS, exist_ok=True)

# --- Presupuesto de computo (evita desconexiones) ---
RUN_DOMAINS=["potato_lab"]   # <-- ahora corre el dominio de LABORATORIO (PlantVillage)
GRID_SEEDS=SEEDS[:5]     # 5 semillas para la grilla; sube a 10 solo en la comparacion clave

# Cultivos/datasets de trabajo. Cada entrada: nombre -> carpeta con subcarpetas por clase.
# TODO: ajusta a las rutas que produce tu celda de descarga (sección 2).
DOMAINS = {
    "potato_lab": "/kaggle/working/data/potato_lab",  # PlantVillage (LAB, controlado, redundante)
    "potato":     "/kaggle/working/data/potato",      # Shabrina + Sankalana (CAMPO)
    "tomato":     "/kaggle/working/data/tomato",      # (opcional) campo tomate
    "cassava":    "/kaggle/working/data/cassava",     # (opcional) campo cassava
}
OOD_TEST = "cassava"     # dataset reservado para evaluación fuera de distribución (RQ2)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

## Como correrlo SIN que se desconecte (Kaggle)
1. Sube a kaggle.com/code -> New Notebook. Settings: **Accelerator=GPU (T4 x2)** e **Internet=ON**.
2. Corre celdas 0-8 una vez para verificar descargas.
3. Luego **Save Version -> Save & Run All (Commit)**: Kaggle ejecuta TODO en segundo plano (hasta 12h).
   Puedes cerrar el navegador; al terminar los CSV quedan en Output.
4. Por defecto RUN_DOMAINS=['potato'] y GRID_SEEDS=5 para que entre en una sesion. Corre un cultivo por
   commit y junta los CSV al final.


## 2. Descarga y unificación de datasets (multi-cultivo)
Descarga con `kagglehub` y organiza cada cultivo en `DOMAINS[nombre]/<clase>/*.jpg`.
Reusa tu lógica de mapeo de clases (como en el paper anterior). Deja cada cultivo con un esquema de
clases común y documentado (p.ej. {healthy, early/leaf-blight, late/otra}). **TODO: completar el
mapeo por dataset.**

In [ ]:
import kagglehub, glob, shutil
def link_class(src_glob, dst):
    os.makedirs(dst, exist_ok=True)
    for x in glob.glob(src_glob, recursive=True):
        if os.path.isfile(x) and x.lower().endswith((".jpg",".jpeg",".png")):
            try: os.link(x, os.path.join(dst, os.path.basename(x)))
            except Exception: shutil.copy(x, dst)

def dset_path(slug):
    """En commit mode los datasets adjuntos viven en /kaggle/input/<ultimo-segmento>.
    Usa esa ruta si existe; si no, cae a kagglehub (modo interactivo)."""
    base=slug.split("/")[-1]
    cand=os.path.join("/kaggle/input", base)
    if os.path.isdir(cand):
        return cand
    return kagglehub.dataset_download(slug)


# --- Ejemplo POTATO (Shabrina campo + Sankalana healthy) ---
pf = dset_path("warcoder/potato-leaf-disease-dataset")
pe = dset_path("nirmalsankalana/potato-leaf-healthy-and-late-blight")
link_class(f"{pf}/**/Fungi/**/*",        f"{DOMAINS['potato']}/early_blight")
link_class(f"{pf}/**/Phytopthora/**/*",  f"{DOMAINS['potato']}/late_blight")
link_class(f"{pf}/**/Healthy/**/*",      f"{DOMAINS['potato']}/healthy")
link_class(f"{pe}/**/Healthy/**/*",      f"{DOMAINS['potato']}/healthy")

# --- POTATO LAB (PlantVillage: mismas 3 clases, condiciones controladas -> alta redundancia) ---
pv = dset_path("arjuntejaswi/plant-village")
link_class(f"{pv}/**/Potato___Early_blight/**/*", f"{DOMAINS['potato_lab']}/early_blight")
link_class(f"{pv}/**/Potato___Late_blight/**/*",  f"{DOMAINS['potato_lab']}/late_blight")
link_class(f"{pv}/**/Potato___healthy/**/*",      f"{DOMAINS['potato_lab']}/healthy")

# --- TODO: TOMATE y CASSAVA (repite el patrón con sus keys de Kaggle y mapeo de clases) ---
# tomate:  dset_path("<owner/tomato-field-dataset>")
# cassava: dset_path("<owner/cassava-leaf-disease>")

for name, root in DOMAINS.items():
    if os.path.isdir(root):
        for c in sorted(os.listdir(root)):
            n=len(glob.glob(f"{root}/{c}/*"))
            print(f"{name}/{c}: {n}")

## 3. Embeddings DINOv2 + pHash (sin circularidad: no usa el backbone del clasificador)

In [ ]:
import timm, imagehash
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

dino = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0).to(DEVICE).eval()
dino_tf = T.Compose([T.Resize((518,518)), T.ToTensor(), T.Normalize(MEAN,STD)])

def list_domain(name):
    root=DOMAINS[name]; files=[]; labels=[]; classes=sorted(os.listdir(root))
    for ci,c in enumerate(classes):
        for x in glob.glob(f"{root}/{c}/*"):
            if os.path.isfile(x): files.append(x); labels.append(ci)
    return files, np.array(labels), classes

class ImgDS(Dataset):
    def __init__(s,f,tf): s.f=f; s.tf=tf
    def __len__(s): return len(s.f)
    def __getitem__(s,i): return s.tf(Image.open(s.f[i]).convert("RGB"))

@torch.no_grad()
def dino_embed(files):
    Z=[]
    for xb in DataLoader(ImgDS(files,dino_tf), batch_size=64, num_workers=2):
        Z.append(dino(xb.to(DEVICE)).cpu().numpy())
    Z=np.concatenate(Z,0); Z=Z/ (np.linalg.norm(Z,axis=1,keepdims=True)+1e-8)
    return Z

def phash_bits(files):
    return np.array([imagehash.phash(Image.open(f).convert("RGB")).hash.flatten() for f in files], dtype=np.float32)

## 4. Clustering aglomerativo a varios umbrales τ (define la granularidad de redundancia)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
def cluster_at(Z, tau):
    return AgglomerativeClustering(n_clusters=None, distance_threshold=tau,
                                   metric="cosine", linkage="average").fit_predict(Z)
# Precompute por dominio: embeddings + clusters a cada tau
DOM = {}
for name in DOMAINS:
    if not os.path.isdir(DOMAINS[name]): continue
    files, labels, classes = list_domain(name)
    print(f"[{name}] {len(files)} imgs, clases={classes}")
    Z = dino_embed(files)
    clusters = {tau: cluster_at(Z, tau) for tau in TAUS}
    import pandas as pd
    diag=[]
    for tau in TAUS:
        cl=clusters[tau]; sizes=np.bincount(cl); sing=100.0*np.mean(sizes[cl]==1)
        print(f"   tau={tau}: {len(set(cl))} clusters | max={sizes.max()} mean={sizes.mean():.2f} singletons={sing:.0f}%")
        diag.append(dict(domain=name,tau=tau,n_clusters=int(len(set(cl))),
                         max_size=int(sizes.max()),mean_size=float(sizes.mean()),singleton_pct=float(sing)))
    pd.DataFrame(diag).to_csv(f"{RESULTS}/clusters_diag_{name}.csv",index=False)
    DOM[name] = dict(files=files, labels=labels, classes=classes, Z=Z, clusters=clusters)

## 5. Splits (aleatorio vs VC-GKF) y muestreo Redundancy-Aware

In [ ]:
from sklearn.model_selection import train_test_split
def random_split(labels, seed):
    idx=np.arange(len(labels))
    tr,tmp=train_test_split(idx,test_size=.3,random_state=seed,stratify=labels)
    va,te =train_test_split(tmp,test_size=.5,random_state=seed,stratify=labels[tmp])
    return tr,va,te
def vcgkf_split(clabels, seed):
    idx=np.arange(len(clabels)); uniq=np.array(sorted(set(clabels)))
    rng=np.random.default_rng(seed); rng.shuffle(uniq); n=len(uniq)
    c_tr,c_va,c_te=uniq[:int(.7*n)],uniq[int(.7*n):int(.85*n)],uniq[int(.85*n):]
    return (idx[np.isin(clabels,c_tr)], idx[np.isin(clabels,c_va)], idx[np.isin(clabels,c_te)])
def redundancy_weights(clabels, train_idx):
    # peso ∝ 1/tamaño de clúster -> down-weight de comunidades redundantes
    sizes=np.bincount(clabels)
    w=np.array([1.0/sizes[clabels[i]] for i in train_idx]); return w/w.sum()*len(w)

## 6. Modelos, pérdidas y bucle de entrenamiento

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torchvision import models
from torch.utils.data import Subset, WeightedRandomSampler
train_tf=T.Compose([T.Resize((IMG,IMG)),T.RandomHorizontalFlip(),T.RandomVerticalFlip(),
                    T.RandomRotation(20),T.ColorJitter(.1,.1,.1),T.ToTensor(),T.Normalize(MEAN,STD)])
eval_tf =T.Compose([T.Resize((IMG,IMG)),T.ToTensor(),T.Normalize(MEAN,STD)])
class LabeledDS(Dataset):
    def __init__(s,files,labels,tf): s.f=files; s.y=labels; s.tf=tf
    def __len__(s): return len(s.f)
    def __getitem__(s,i): return s.tf(Image.open(s.f[i]).convert("RGB")), int(s.y[i])
def build(key,nc):
    if key=="resnet50": m=models.resnet50(weights="IMAGENET1K_V2"); m.fc=nn.Linear(m.fc.in_features,nc)
    elif key=="mobilenetv3": m=models.mobilenet_v3_large(weights="IMAGENET1K_V2"); m.classifier[3]=nn.Linear(m.classifier[3].in_features,nc)
    elif key=="efficientnetv2": m=models.efficientnet_v2_s(weights="IMAGENET1K_V1"); m.classifier[1]=nn.Linear(m.classifier[1].in_features,nc)
    elif key=="vit": m=models.vit_b_16(weights="IMAGENET1K_V1"); m.heads.head=nn.Linear(m.heads.head.in_features,nc)
    return m.to(DEVICE)
class Focal(nn.Module):
    def __init__(s,alpha,g=GAMMA): super().__init__(); s.a=alpha; s.g=g
    def forward(s,z,y):
        logp=F.log_softmax(z,1); p=logp.exp()
        lt=logp.gather(1,y[:,None]).squeeze(1); pt=p.gather(1,y[:,None]).squeeze(1); at=s.a.gather(0,y)
        return (-at*(1-pt)**s.g*lt).mean()
def alpha_for(labels_train,nc):
    c=np.bincount(labels_train,minlength=nc).astype(float); inv=1/np.clip(c,1,None)
    return torch.tensor(inv/inv.sum()*nc,dtype=torch.float32,device=DEVICE)

def train_eval(key, files, labels, tr, va, te, seed, loss="focal", redund_w=None):
    set_seed(seed); nc=len(set(labels))
    dl_tr=DataLoader(LabeledDS([files[i] for i in tr],labels[tr],train_tf),batch_size=BATCH,
                     sampler=(WeightedRandomSampler(redund_w,len(redund_w),replacement=True) if redund_w is not None else None),
                     shuffle=(redund_w is None), num_workers=2, pin_memory=True)
    dl_te=DataLoader(LabeledDS([files[i] for i in te],labels[te],eval_tf),batch_size=BATCH,num_workers=2)
    m=build(key,nc); crit=Focal(alpha_for(labels[tr],nc)) if loss=="focal" else nn.CrossEntropyLoss(weight=alpha_for(labels[tr],nc))
    opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=WD)
    m.train()
    for _ in range(EPOCHS):
        for x,y in dl_tr:
            x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(); crit(m(x),y).backward(); opt.step()
    m.eval(); P=[]; Y=[]; PR=[]
    with torch.no_grad():
        for x,y in dl_te:
            o=m(x.to(DEVICE)); PR.append(torch.softmax(o,1).cpu().numpy()); P.append(o.argmax(1).cpu().numpy()); Y.append(y.numpy())
    import numpy as _np; P=_np.concatenate(P); Y=_np.concatenate(Y); PR=_np.concatenate(PR)
    from sklearn.metrics import accuracy_score,f1_score
    return dict(acc=accuracy_score(Y,P), f1=f1_score(Y,P,average="macro")), (m if False else None), (Y,P,PR)

## Experimento A — Grilla principal (modelos × splits × 10 semillas) por cultivo
Produce `gridA.csv`: acc/F1 por (dominio, modelo, split, semilla). Base para el flip de ranking y el LSI.

In [ ]:
import pandas as pd
rows=[]
for dname in [d for d in DOM if d in RUN_DOMAINS]:
    d=DOM[dname]; files,labels=d["files"],d["labels"]
    for model in MODELS:
        for seed in GRID_SEEDS:
            # split aleatorio
            tr,va,te=random_split(labels,seed)
            mA,_,_=train_eval(model,files,labels,tr,va,te,seed)
            rows.append(dict(domain=dname,model=model,split="random",tau=None,seed=seed,**mA))
            # split VC-GKF a tau=0.15 (principal)
            tr,va,te=vcgkf_split(d["clusters"][0.15],seed)
            mB,_,_=train_eval(model,files,labels,tr,va,te,seed)
            rows.append(dict(domain=dname,model=model,split="vcgkf",tau=0.15,seed=seed,**mB))
            print(dname,model,seed,"acc_rand=%.3f acc_vc=%.3f"%(mA["acc"],mB["acc"]))
pd.DataFrame(rows).to_csv(f"{RESULTS}/gridA.csv",index=False)

## Experimento B — Curva LSI (barrido de τ) y LSI escalar
Produce `lsi_curve.csv` y `lsi.csv`. Entrena VC-GKF a cada τ (semilla 42, todos los modelos) y calcula
el LSI puntual y de curva (área normalizada) por (dominio, modelo).

In [ ]:
from numpy import trapz
rowsC=[]; rowsL=[]
for dname in [d for d in DOM if d in RUN_DOMAINS]:
    d=DOM[dname]; files,labels=d["files"],d["labels"]
    for model in MODELS:
        tr,va,te=random_split(labels,42); acc0=train_eval(model,files,labels,tr,va,te,42)[0]["acc"]
        accs=[]
        for tau in TAUS:
            tr,va,te=vcgkf_split(d["clusters"][tau],42); a=train_eval(model,files,labels,tr,va,te,42)[0]["acc"]
            accs.append(a); rowsC.append(dict(domain=dname,model=model,tau=tau,acc=a,acc_random=acc0))
        lsi_curve=trapz([acc0-a for a in accs],TAUS)/(max(TAUS)*acc0)
        lsi0=(acc0-accs[TAUS.index(0.15)])/acc0
        rowsL.append(dict(domain=dname,model=model,LSI_point=lsi0,LSI_curve=lsi_curve,acc_random=acc0))
        print(dname,model,"LSI0=%.3f LSIcurve=%.3f"%(lsi0,lsi_curve))
pd.DataFrame(rowsC).to_csv(f"{RESULTS}/lsi_curve.csv",index=False)
pd.DataFrame(rowsL).to_csv(f"{RESULTS}/lsi.csv",index=False)

## Experimento C — Evaluación OOD cross-dataset (RQ2: ¿el LSI predice la caída OOD?)
Entrena en cada cultivo (VC-GKF) y evalúa en el dataset OOD reservado. `ood.csv`.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
def eval_on(model_state_files, labels_te_files, labels_te):
    pass  # placeholder si guardas checkpoints; aquí reentrenamos por simplicidad
rowsO=[]
ood=DOM.get(OOD_TEST)
for dname in [d for d in DOM if d in RUN_DOMAINS]:
    if dname==OOD_TEST or ood is None: continue
    d=DOM[dname]
    for model in MODELS:
        tr,va,te=vcgkf_split(d["clusters"][0.15],42)
        # reentrena y evalúa en OOD (clases deben mapear; ajusta si difieren)
        _, _, _ = train_eval(model,d["files"],d["labels"],tr,va,np.arange(len(ood["labels"])),42)
        # NOTA: para OOD real, guarda el modelo y evalúa sobre ood["files"]. Ver comentario abajo.
        rowsO.append(dict(train_domain=dname,model=model,ood_domain=OOD_TEST))
        print("OOD", dname,"->",OOD_TEST, model)
# Para OOD correcto: modifica train_eval para devolver el modelo y evalúalo sobre ood["files"]/labels.
pd.DataFrame(rowsO).to_csv(f"{RESULTS}/ood.csv",index=False)

## Experimento D — Mitigación Redundancy-Aware (RQ4)
Compara entrenamiento estándar vs muestreo balanceado por clúster, evaluado en VC-GKF y OOD. `mitig.csv`.

In [ ]:
rowsM=[]
for dname in [d for d in DOM if d in RUN_DOMAINS]:
    d=DOM[dname]; files,labels=d["files"],d["labels"]
    for model in MODELS:
        for seed in GRID_SEEDS:
            tr,va,te=vcgkf_split(d["clusters"][0.15],seed)
            base=train_eval(model,files,labels,tr,va,te,seed)[0]
            w=redundancy_weights(d["clusters"][0.15],tr)
            ra=train_eval(model,files,labels,tr,va,te,seed,redund_w=w)[0]
            rowsM.append(dict(domain=dname,model=model,seed=seed,
                              acc_base=base["acc"],acc_redund=ra["acc"],
                              f1_base=base["f1"],f1_redund=ra["f1"]))
            print(dname,model,seed,"base=%.3f redund=%.3f"%(base["acc"],ra["acc"]))
pd.DataFrame(rowsM).to_csv(f"{RESULTS}/mitig.csv",index=False)

## Experimento E — Estadística (Wilcoxon n=10 + tamaño de efecto + bootstrap)
Sobre `gridA.csv`: inflación random vs vcgkf por modelo/dominio, con p-value, Cliff's δ e IC bootstrap.

In [ ]:
from scipy.stats import wilcoxon
g=pd.read_csv(f"{RESULTS}/gridA.csv"); rng=np.random.default_rng(0); rowsE=[]
def cliffs_delta(a,b):
    a=np.array(a); b=np.array(b); gt=sum((x>y) for x in a for y in b); lt=sum((x<y) for x in a for y in b)
    return (gt-lt)/(len(a)*len(b))
for dname in g.domain.unique():
    for model in MODELS:
        r=g[(g.domain==dname)&(g.model==model)&(g.split=="random")].sort_values("seed").acc.values
        v=g[(g.domain==dname)&(g.model==model)&(g.split=="vcgkf")].sort_values("seed").acc.values
        if len(r)<2: continue
        try: W,p=wilcoxon(r,v)
        except ValueError: W,p=0.0,1.0
        diff=(r-v)*100; boots=[rng.choice(diff,len(diff),replace=True).mean() for _ in range(10000)]
        rowsE.append(dict(domain=dname,model=model,gap_pp=diff.mean(),
                          p_value=p,cliffs_delta=cliffs_delta(r,v),
                          ci_lo=np.percentile(boots,2.5),ci_hi=np.percentile(boots,97.5)))
pd.DataFrame(rowsE).to_csv(f"{RESULTS}/stats.csv",index=False)
print(pd.DataFrame(rowsE))

## Experimento F — Empaquetar resultados
Comprime `RESULTS/` para descargar los resultados.

In [ ]:
import shutil
shutil.make_archive("resultados","zip",RESULTS)
print("Listo: resultados.zip")
print(os.listdir(RESULTS))

## Notas de implementación
- **XAI ratio de energía (4 modelos):** reusa `xai_energy_ratio_cpu.py` con los checkpoints que guardes
  aquí (añade `torch.save(m.state_dict(),...)` en `train_eval`). Corre en CPU.
- **OOD correcto:** modifica `train_eval` para *devolver el modelo* y evaluarlo sobre `ood["files"]`
  (mapea las clases al esquema común). El esqueleto deja el gancho.
- **Presupuesto:** si el cómputo aprieta, reduce a 3 modelos o baja SEEDS de la grilla a 5 (deja 10 solo
  para la comparación clave ResNet/MobileNet vs ViT).